# ML Masterclass 02: Linear Models & Optimization

In this module, we dissect the workhorses of statistical learning. Linear models are the foundation of industry-grade AI, providing high-speed, interpretable predictions for high-stakes domains like finance, insurance, and medical diagnostics.

---

## Table of Contents
1. **Linear & Logistic Regression:** Analytical solutions vs Iterative Gradient Descent.
2. **ElasticNet:** The robust compromise between L1 and L2 regularization.
3. **Generalized Linear Models (GLMs):** The unifying framework for Poisson, Bernoulli, and Gaussian data.
4. **Convergence Theory:** The mathematical conditions ($L$-Lipschitz, Strong Convexity) for guaranteed optimization.

---

## 🎓 Socratic Deep Check
**The Efficiency vs Precision Trade-off:**
> *"The Normal Equation gives the exact, analytically perfect answer for Linear Regression. Why does the entire AI industry instead use 'imprecise' Gradient Descent for 100-layer Neural Networks? What is the $O(N^3)$ wall that kills analytical math?"*

---
📈 **Production Signal:** Logistic regression powers ad-click prediction at Meta/Google scale (billions of requests per second), as it is one of the few models capable of 1ms latency under heavy load.

---


### 🎓 Junior to Senior: Concept Bridge

**The Senior Perspective:** You don't just use a linear model because it's 'simple'. You use it because it's **identifiable**. In a GLM, every parameter has a direct physical meaning (e.g., 'Increasing temperature by 1° increases claim frequency by 5%'). In production, interpretability is often more valuable than a 1% gain in accuracy.

**Mental Model:** 
- **Linear Regression:** A best-fit line through a cloud of data points.
- **ElasticNet:** A 'smart' regularity filter that handles redundant features much better than Lasso alone.
- **GLMs:** A single mathematical machine that can be 'configured' to handle counts (Poisson), binary flips (Logistic), or ratings (Regression) by just swapping a 'Link Function'.

**Common Junior Pitfall:** Using MSE for count data (e.g., number of website clicks). Count data must be non-negative and follows a Poisson distribution. Using MSE assumes the noise can be negative, leading to physically impossible predictions (e.g., '-2 people visited the site today').

---


## 1. Linear & Logistic Regression
We aim to find $W$ that minimizes the loss surface.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

np.random.seed(42)
X = 2 * np.random.rand(100, 1)
y = 4 + 3 * X + np.random.randn(100, 1)
X_b = np.c_[np.ones((100, 1)), X]

def GD(X, y, lr=0.1, n=1000):
    m = len(y); w = np.random.randn(2, 1)
    for _ in range(n):
        grad = 2/m * X.T.dot(X.dot(w) - y)
        w -= lr * grad
    return w

print(f"Normal Equation result: {np.linalg.inv(X_b.T.dot(X_b)).dot(X_b.T).dot(y).flatten()}")
print(f"Gradient Descent result: {GD(X_b, y).flatten()}")

"""
What just happened?
We compared the exact analytical solution to the iterative approach. For 100 samples,
the Normal Equation is instant. But for massive datasets, GD is the standard.
"""

## 2. ElasticNet — The Regularization Compromise

Lasso (L1) deletes features. Ridge (L2) shrinks them. 
**ElasticNet** combines them:
$$L_{EN} = L + \lambda_1 ||w||_1 + \lambda_2 ||w||_2^2$$

**Why use it?**
If you have two highly correlated features ($X_1, X_2$), Lasso might arbitrarily keep $X_1$ and delete $X_2$. This makes the model unstable if $X_2$ was actually the causal factor. ElasticNet keeps both but shares the weight, providing the "grouping effect."

### Coordinate Descent
ElasticNet is often optimized using **Coordinate Descent**: we optimize one weight at a time while holding others fixed. This allows us to use the **Soft Thresholding** operator to find the precise zero-point for each feature.


In [ ]:
def soft_threshold(z, gamma):
    return np.sign(z) * np.maximum(np.abs(z) - gamma, 0)

def elastic_net_coordinate_descent(X, y, l1=1.0, l2=1.0, n_iter=100):
    n_samples, n_features = X.shape
    w = np.zeros(n_features)
    
    for _ in range(n_iter):
        for j in range(n_features):
            # Calculate partial residual
            r_j = y - (X @ w - X[:, j] * w[j])
            rho_j = X[:, j] @ r_j
            
            # Update weight using soft thresholding
            denominator = (X[:, j] @ X[:, j]) + l2
            w[j] = soft_threshold(rho_j, l1) / denominator
    return w

# Simulated correlated data
X_corr = np.array([[1, 0.9, 0], [0.9, 1, 0], [0, 0, 1]])
y_corr = np.array([2, 2, 5])
w_en = elastic_net_coordinate_descent(X_corr, y_corr, l1=0.1, l2=0.1)
print(f"ElasticNet Weights: {w_en} (Notice X[0] and X[1] are shared!)")

"""
What just happened?
We implemented Coordinate Descent. For each feature, we calculated how much it 
contributes to the error and applied a 'dead zone' (Soft Thresholding). If its 
contribution isn't stronger than the penalty l1, the weight drops to exactly 0.
"""

## 3. Generalized Linear Models (GLMs)

A GLM is a unifying framework for all regression. It consists of:
1. **Random Component:** The distribution family (Gaussian, Bernoulli, Poisson).
2. **Systematic Component:** The linear predictor $\eta = Xw$.
3. **Link Function:** Maps $\eta$ to the predicted mean. e.g. $\lambda = \exp(\eta)$.

### Poisson Regression (Count Data)
For predicting counts ($0, 1, 2, \dots$), we use the Poisson NLL loss:
$$L = -\sum (y_i \eta_i - \exp(\eta_i))$$
where $\exp(\eta_i)$ ensures our predicted intensity $\lambda$ is always positive.


In [ ]:
def poisson_regression_gd(X, y, lr=0.01, n=5000):
    w = np.zeros((X.shape[1], 1))
    for i in range(n):
        # λ = exp(Xw)
        y_hat = np.exp(X.dot(w))
        # Gradient of NLL: X^T (y_hat - y)
        grad = X.T.dot(y_hat - y)
        w -= lr * grad / X.shape[0]
    return w

# Predict claim frequency based on risk scores
X_risk = np.array([[1, 0.5], [1, 2.0], [1, 5.0]]) # [Intercept, Risk]
y_risk = np.array([[1], [3], [10]]) # Number of claims
w_poisson = poisson_regression_gd(X_risk, y_risk)
print(f"Poisson Weights: {w_poisson.flatten()}")

"""
What just happened?
We implemented a counting model. Even though the input risk scores are small,
the exponential link function allows the predictions to grow exponentially to 
match the high claim counts.
"""

## 4. Convergence Theory — The Mathematics of Speed

Why doesn't Gradient Descent take a billion years to finish? 

**L-Lipschitz Gradients:** If the gradient doesn't change faster than some constant $L$, then with $lr = 1/L$, GD is guaranteed to decrease the loss at each step: $O(1/n)$.

**Strong Convexity ($\mu$):** If the loss surface is 'pointy' enough, we get **Exponential Convergence**:
$$||w_t - w^*|| \le \left(1 - \frac{\mu}{L}\right)^t ||w_0 - w^*||$$

🎓 **Socratic Deep Check:**
> *"If you add massive L2 Regularization to your model, you are physically INCREASING the strong convexity constant $\mu$. Why does this make your model converge MUCH faster, even though it might decrease your peak accuracy?"*

In [ ]:
L, mu = 100.0, 1.0
iterations = np.arange(100)
# k = (1 - mu/L)
theory_slope = np.log(1 - mu/L)

# Simulate loss decay for a strongly convex function
loss_decay = (1 - mu/L)**iterations

plt.plot(iterations, np.log(loss_decay))
plt.title(f"Exponential Convergence: Log-Loss Slope is {theory_slope:.4f}")
plt.xlabel("Iterations"); plt.ylabel("Log Loss")
plt.show()

"""
What just happened?
We visualized the convergence speed guarantee. On a log-scale, strongly convex 
convergence is a perfectly straight line, proving that every step takes us 
a fixed percentage closer to the truth.
"""

---
## ✅ Knowledge Check

### Q1: Why use GLM for Insurance?
<details><summary>👉 Answer</summary>
Insurance features are often multiplicative ('Adding a car alarm reduces theft risk by 20%'). GLMs with a log-link (Poisson) automatically transform these additive linear weights into multiplicative factors ($\exp(w) \cdot \exp(x)$), which perfectly matches the business logic.
</details>

### Q2: ElasticNet vs Lasso/Ridge
<details><summary>👉 Answer</summary>
Lasso is a 'shark' — it kills features aggressively. Ridge is a 'shifter' — it spreads weight equally. ElasticNet is the middle ground; it provides the sparsity of Lasso while maintaining the stability and grouping effect of Ridge for correlated data.
</details>


---
## 🔨 Practical Practice

### Tier 1: Beginner
1. Implement the **Soft Thresholding** operator and verify that inputs between $[-\gamma, \gamma]$ are set to exactly 0.
2. Modify `poisson_regression_gd` to print the predicted $\lambda$ for a user with Risk=10.0. Why does it not crash even with huge inputs?

### Tier 2: Intermediate
1. **The Multi-Collinearity Stress Test:** Create a dataset where $X_1$ and $X_2$ are 99% correlated. Train both `Lasso` and `ElasticNet`. Show that Lasso picks one randomly, while ElasticNet keeps both.
2. **Convergence Benchmarking:** Run GD on a quadratic surface with $L=10$ and $L=100$. Plot the loss curves and verify that the one with lower $L$ (flatter surface) allows for a larger $\eta$ and faster convergence.

### Tier 3: Advanced
1. **Gamma Regression from Scratch:** Using the GLM framework, implement **Gamma Regression** (link function: inverse) for predicting strictly positive, long-tailed data (like 'Time to Failure' in manufacturing).
2. **Coordinate Descent Complexity:** Prove that the time complexity of one full epoch of Coordinate Descent is $O(N \cdot K)$ (where $N$=samples, $K$=features). Why is this better than the $O(K^3)$ of the Normal Equation for extremely wide data?
